# NOTEBOOK: 00 — Pipeline Orchestrator

- **Layer:** All layers — Bronze → Silver → Gold
- **Purpose:** Execute the complete TechMart Catalog Intelligence Pipeline end-to-end in a single run.
- **Stages:** Bronze Ingestion → Silver Standardization → LLM Extraction → LLM Judge → Silver Taxonomy → Gold Aggregation
- **Notes:** 
  - Run this notebook to trigger the full pipeline. 
  - Each stage calls its notebook via `dbutils.notebook.run()`. 
  - A 90-second pause between Stage 3 and Stage 4 respects Groq API rate limits.
- **Author:** Maira Tavares

In [0]:
print("╔══════════════════════════════════════════════════════╗")
print("║   TechMart Catalog Intelligence Pipeline             ║")
print("║   End-to-End Orchestrator                            ║")
print("╚══════════════════════════════════════════════════════╝")

# 1. Bronze Ingestion

In [0]:
# Bronze: Raw ingestion
# Reads raw Excel files from Unity Catalog Volume
# Writes raw_products and raw_vendors Delta tables

print("=" * 55)
print("Stage 1 — Bronze Ingestion")
print("=" * 55)

dbutils.notebook.run(f"01_bronze_ingest", timeout_seconds = 300)

print("✅ Stage 1 complete: Bronze ingestion complete")

#  2. Silver Standardization

In [0]:
# Silver: Standardization
# Normalizes weight, price, vendor names
# Writes silver products and vendors Delta tables

print("=" * 55)
print("Stage 2 — Silver Standardization")
print("=" * 55)

dbutils.notebook.run("02_silver_standardize", timeout_seconds=300)

print("✅ Stage 2 complete: Silver standardization complete")

#  3. LLM Extraction

In [0]:
# Silver: LLM Extraction
# Calls Groq API to extract name, brand, sub-category
# Logs results to MLflow for traceability

print("=" * 55)
print("Stage 3 — LLM Extraction")
print("=" * 55)

dbutils.notebook.run("03_llm_extraction", timeout_seconds=600)

print("✅ Stage 3 complete: LLM extraction complete")

In [0]:
# After Stage 3
print("⏳ Waiting 90 seconds for API quota reset...")
import time
time.sleep(90)


# 4. LLM Judge

In [0]:
# Silver: LLM Judge
# Validates extraction against approved taxonomy
# Flags records that don't meet quality standards

print("=" * 55)
print("Stage 4 — LLM Judge")
print("=" * 55)

dbutils.notebook.run("./04_llm_judge", timeout_seconds=600)

print("✅ Stage 4 complete: LLM judge complete")

# 5. Silver Taxonomy Enrichment

In [0]:
# Silver: Taxonomy Enrichment
# Joins LLM results with vendor info and category mapping

print("=" * 55)
print("Stage 5 — Silver Taxonomy Enrichment")
print("=" * 55)

dbutils.notebook.run("05_silver_taxonomy", timeout_seconds=300)

print("✅ Stage 5 complete: Silver taxonomy enrichment complete")

# 6. Gold Aggregation

In [0]:
# Gold: Aggregation
# Produces final business-ready aggregated table

print("=" * 55)
print("Stage 6 — Gold Aggregation")
print("=" * 55)

dbutils.notebook.run("06_gold_aggregation", timeout_seconds=300)

print("✅ Stage 6 complete: Gold aggregation complete")

# Summary

In [0]:
from utils.config import (
    BRONZE_PRODUCTS,
    BRONZE_VENDORS, 
    SILVER_TAXONOMY,
    SILVER_VENDORS,
    SILVER_PRODUCTS,
    TAXONOMY_ENRICHED,
    LLM_EXTRACTED,
    GOLD_SUMMARY,
)

# Displays row counts for all Delta tables produced
# by the pipeline — confirms end-to-end execution

tables = {
    "Bronze — raw_products"      : BRONZE_PRODUCTS,
    "Bronze — raw_vendors"       : BRONZE_VENDORS,
    "Silver — products"          : SILVER_PRODUCTS,
    "Silver — vendors"           : SILVER_VENDORS,
    "Silver — llm_extracted"     : LLM_EXTRACTED,
    "Silver — taxonomy"          : SILVER_TAXONOMY,
    "Silver — taxonomy_enriched" : TAXONOMY_ENRICHED,
    "Gold   — product_summary"   : GOLD_SUMMARY,
}

print("╔══════════════════════════════════════════════════════╗")
print("║   Pipeline Complete — Final Summary                  ║")
print("╚══════════════════════════════════════════════════════╝")
print(f"  {'Table':<35} {'Rows':>6}")
print("  " + "-" * 43)

all_ok = True
for label, table in tables.items():
    try:
        count = spark.table(table).count()
        print(f"  {label:<35} {count:>6} rows")
    except Exception as e:
        print(f"  {label:<35} ⚠️  not found")
        all_ok = False

print("  " + "-" * 43)
if all_ok:
    print("  ✅ All tables verified — pipeline complete")
else:
    print("  ⚠️  Some tables missing — check logs above")